<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# LogiTrack — Supply Chain Analytics
## Notebook 5 — Machine Learning Premium : Prédiction des retards
### 📝 VERSION APPRENANT

> **Instructions :** Complète les cellules marquées `# TODO`.  
> Les blocs `MÉTHODE` t'expliquent l'approche attendue.  
> Les blocs `🧠 Tes observations` sont à remplir après exécution.

| | |
|---|---|
| **Prérequis** | `logitrack_analytics.csv` dans `SAVE_PATH` (produit par NB2) |
| **Niveau** | Premium |
| **Outils** | Python — scikit-learn, pandas, matplotlib |
| **Durée estimée** | 3h à 4h |

> **Problème :** Classification binaire — prédire `livraison_en_retard = 1` avant l'expédition.
>
> **Contrainte métier :** Recall > 0.75. Il vaut mieux alerter par excès (fausse alarme) que rater un vrai retard. Le coût d'une fausse alerte (dispatcher qui vérifie un colis qui arrivera à temps) est bien inférieur au coût d'un retard non détecté (pénalité + client mécontent).
>
> **Coupure temporelle :** Train avant 2023-07-01 / Test après 2023-07-01.

---
## Étape 1 — Imports et configuration

> **MÉTHODE — Pourquoi `TimeSeriesSplit` et non `KFold` ?**
>
> Les données de livraison sont temporelles — une livraison créée en janvier 2023 ne peut pas être utilisée pour prédire une livraison de décembre 2022. `KFold` ignore l'ordre temporel et crée du **leakage** : le modèle apprend sur le futur pour prédire le passé. `TimeSeriesSplit` garantit que chaque fold de test est toujours plus récent que le fold d'entraînement.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, sys

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

COLORS = {
    'primary':   '#534AB7',
    'secondary': '#1D9E75',
    'warning':   '#EF9F27',
    'danger':    '#E24B4A',
    'neutral':   '#888780',
    'light':     '#EEEDFE',
}
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F9F9F8',
    'axes.grid':        True,
    'grid.alpha':       0.35,
    'font.size':        11,
})

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, f1_score, recall_score, precision_score,
    confusion_matrix, classification_report
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/DataProjectLab/projects/logitrack/'
else:
    SAVE_PATH = './outputs/'
os.makedirs(SAVE_PATH, exist_ok=True)
print(f'📁 Environnement : {"Colab" if IN_COLAB else "Local"}')
print(f'📁 Dossier       : {SAVE_PATH}')
print('Configuration chargée ✅')

---
## Étape 2 — Chargement et vérification

### MÉTHODE
On vérifie la distribution de la variable cible et les nulls dans les features ML avant toute modélisation. Un dataset déséquilibré (< 20% de classe positive) nécessitera `class_weight='balanced'` ou SMOTE. Les nulls résiduels seront imputés par la médiane — stratégie robuste pour des features numériques.

In [ ]:
BASE_URL   = 'https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/logitrack/data/'
clean_path = f'{SAVE_PATH}logitrack_analytics.csv'

if not os.path.exists(clean_path):
    clean_path = BASE_URL + 'logitrack_analytics.csv'
    print('⚠️  logitrack_analytics.csv non trouvé en local — chargement depuis GitHub')
else:
    print(f'✅ Fichier trouvé : {clean_path}')

df = pd.read_csv(clean_path, parse_dates=['date_creation','date_enlevement','date_livraison_prevue'])

# Exclure transporteur suspendu
df = df[df['transporteur_actif'] == 1].copy()
print(f'Dataset : {df.shape}')

# Distribution variable cible
print(f'\nDistribution livraison_en_retard :')
vc = df['livraison_en_retard'].value_counts()
print(vc)
print(f'  Taux classe positive : {vc[1]/len(df)*100:.1f}%')

# Features ML disponibles
FEATURES = [
    # Temporel
    'mois_sin', 'mois_cos', 'jour_semaine', 'est_weekend',
    'est_fin_mois', 'heure_enlevement', 'delai_prep_heures',
    # Logistique
    'poids_kg', 'volume_m3', 'densite_kg_m3',
    'priorite_num', 'sla_jours_contractuel',
    'distance_km', 'duree_standard_jours', 'nb_points_controle',
    # Performance historique
    'taux_breach_transporteur', 'taux_breach_corridor',
    'taux_breach_entrepot_origine', 'taux_livraison_temps',
    'ponctualite_entrepot_origine',
    # Contexte
    'risque_douanier_num', 'mode_transport_num',
    'segment_num', 'penalite_retard_pct',
]

print(f'\nNulls dans les features :')
nulls = df[FEATURES].isnull().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else '  Aucun null ✅')

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
## Étape 3 — Coupure temporelle

### MÉTHODE
La coupure temporelle est **la règle la plus importante** du ML sur données de flux. En utilisant `date_creation < 2023-07-01` pour le train et `>= 2023-07-01` pour le test, on simule les conditions réelles de production : le modèle est entraîné sur le passé et prédit sur le futur.

**Features exclues pour éviter le data leakage :**
- `delai_jours` — contient le résultat final de la livraison
- `sla_breach`, `escalade`, `in_backlog` — composantes de la variable cible
- `date_livraison_reelle` — inconnue au moment de l'expédition
- `duree_reelle_jours` — inconnue au moment de l'expédition
- `ratio_duree_vs_standard` — calculé depuis `duree_reelle_jours`
- `delai_relatif_sla` — calculé depuis `duree_reelle_jours`

In [ ]:
CIBLE        = 'livraison_en_retard'
DATE_COUPURE = pd.Timestamp('2023-07-01')

df_ml = df[FEATURES + [CIBLE, 'date_creation']].copy()
for col in FEATURES:
    if df_ml[col].isnull().sum() > 0:
        df_ml[col] = df_ml[col].fillna(df_ml[col].median())

# TODO : créer train (date_creation < DATE_COUPURE) et test (>=)
train = # TODO
test  = # TODO
X_train, y_train = train[FEATURES], train[CIBLE]
X_test,  y_test  = test[FEATURES],  test[CIBLE]

print(f'Train : {len(train):,} lignes | taux positifs : {y_train.mean()*100:.1f}%')
print(f'Test  : {len(test):,} lignes  | taux positifs : {y_test.mean()*100:.1f}%')

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
## Étape 4 — Entraînement de 3 modèles

### MÉTHODE
`class_weight='balanced'` pénalise davantage les erreurs sur la classe minoritaire en pondérant inversement les classes par leur fréquence. Même avec un dataset quasi-équilibré (39/61), cela améliore le Recall car le modèle est incité à ne pas ignorer les cas positifs.

**Standardisation :** `StandardScaler` uniquement pour la Logistic Regression — les arbres de décision (RF, GB) sont insensibles à l'échelle des features.

In [ ]:
# Standardisation pour Logistic Regression
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# TODO : Logistic Regression (class_weight='balanced', max_iter=500)
lr = LogisticRegression(# TODO)
lr.fit(X_train_sc, y_train)
y_pred_lr  = lr.predict(X_test_sc)
y_proba_lr = lr.predict_proba(X_test_sc)[:, 1]

# TODO : Random Forest (n_estimators=300, max_depth=8, class_weight='balanced')
rf = RandomForestClassifier(# TODO)
rf.fit(X_train, y_train)
y_pred_rf  = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

# TODO : Gradient Boosting (n_estimators=200, max_depth=4, learning_rate=0.05)
gb = GradientBoostingClassifier(# TODO)
gb.fit(X_train, y_train)
y_pred_gb  = gb.predict(X_test)
y_proba_gb = gb.predict_proba(X_test)[:, 1]

# TODO : construire le tableau comparatif (AUC-ROC, F1, Recall, Precision)
results = pd.DataFrame({
    'Modèle':    ['Logistic Regression', 'Random Forest', 'Gradient Boosting'],
    'AUC-ROC':   [# TODO],
    'F1':        [# TODO],
    'Recall':    [# TODO],
    'Precision': [# TODO],
}).round(4)
print(results.to_string(index=False))

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
## Étape 5 — Optimisation du seuil de décision

### MÉTHODE
L'AUC-ROC mesure la capacité discriminante du modèle sur **tous** les seuils possibles. Un AUC de 0.85 signifie que le modèle classe correctement 85% des paires (retard, à temps). Le seuil par défaut (0.5) n'est pas toujours optimal pour une contrainte métier spécifique.

**Principe :** abaisser le seuil → plus d'alertes → Recall augmente, Precision baisse. On cherche le seuil le plus élevé satisfaisant Recall ≥ 0.75 — pour minimiser les fausses alertes tout en couvrant les vrais retards.

In [ ]:
# TODO : tester les seuils de 0.25 à 0.69 par pas de 0.05 sur y_proba_rf
seuils = np.arange(0.25, 0.70, 0.05)
rows = []
for s in seuils:
    yp = (y_proba_rf >= s).astype(int)
    rows.append({
        'seuil':      round(s, 2),
        'recall':     # TODO,
        'precision':  # TODO (zero_division=0),
        'f1':         # TODO (zero_division=0),
        'nb_alertes': int(yp.sum())
    })
df_s = pd.DataFrame(rows)
print(df_s.round(4).to_string(index=False))

# TODO : choisir le premier seuil qui satisfait Recall >= 0.75
optimal       = df_s[df_s['recall'] >= 0.75].iloc[0]
SEUIL_OPTIMAL = float(optimal['seuil'])
print(f'\nSeuil optimal retenu : {SEUIL_OPTIMAL:.2f}')

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(df_s['seuil'], df_s['recall'],    color=COLORS['danger'],    label='Recall',
        linewidth=2.5, marker='o')
ax.plot(df_s['seuil'], df_s['precision'], color=COLORS['secondary'], label='Precision',
        linewidth=2.5, marker='s')
ax.plot(df_s['seuil'], df_s['f1'],        color=COLORS['primary'],   label='F1-Score',
        linewidth=2, linestyle='--')
ax.axvline(SEUIL_OPTIMAL, color=COLORS['warning'], linestyle='--', linewidth=2,
           label=f'Seuil optimal ({SEUIL_OPTIMAL:.2f})')
ax.axhline(0.75, color=COLORS['danger'], linestyle=':', linewidth=1, alpha=0.7,
           label='Contrainte Recall = 0.75')
ax.set_xlabel('Seuil de décision')
ax.set_ylabel('Score')
ax.set_title('Precision vs Recall selon le seuil — Random Forest LogiTrack',
             fontweight='bold')
ax.legend()
ax.set_xlim(0.23, 0.72)
plt.tight_layout()
plt.savefig(f'{SAVE_PATH}logitrack_precision_recall_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Sauvegardé : {SAVE_PATH}logitrack_precision_recall_curve.png')

---
## Étape 6 — Feature Importance

### MÉTHODE
La feature importance du Random Forest mesure la réduction moyenne d'impureté (Gini) apportée par chaque feature lors des splits. Une feature importante est celle qui discrimine le mieux les livraisons en retard des livraisons à temps. C'est aussi un outil de communication — elle dit aux opérationnels sur quels indicateurs se concentrer.

In [ ]:
# TODO : extraire la feature importance du Random Forest
# Indice : rf.feature_importances_ → pd.Series indexée par FEATURES
fi = # TODO
fi = fi.sort_values(ascending=False)
print('=== FEATURE IMPORTANCE ===')
for feat, imp in fi.items():
    bar = '█' * int(imp * 300)
    print(f'  {feat:<35} {imp:.4f} ({imp*100:.1f}%) {bar}')

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
fi_sorted   = fi.sort_values()
colors_fi   = [
    COLORS['danger']  if imp > 0.18 else
    COLORS['warning'] if imp > 0.10 else
    COLORS['primary'] if imp > 0.05 else
    COLORS['neutral']
    for imp in fi_sorted.values
]
ax.barh(fi_sorted.index, fi_sorted.values, color=colors_fi)
for i, (feat, imp) in enumerate(fi_sorted.items()):
    ax.text(imp + 0.002, i, f'{imp*100:.1f}%', va='center', fontsize=9)
ax.set_xlabel('Importance (%)')
ax.set_title('Feature Importance — Random Forest LogiTrack', fontweight='bold')
ax.set_xlim(0, fi.max() * 1.3)
plt.tight_layout()
plt.savefig(f'{SAVE_PATH}logitrack_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Sauvegardé : {SAVE_PATH}logitrack_feature_importance.png')

---
## Étape 7 — Validation par TimeSeriesSplit

### MÉTHODE
`TimeSeriesSplit(n_splits=5)` crée 5 folds en respectant l'ordre temporel. Un modèle stable doit avoir un écart-type de Recall < 0.10 entre les folds. Un écart-type élevé signale que le modèle est sensible à la période — il faudra alors investiguer si des changements opérationnels (nouveau transporteur, nouvelle route) ont cassé la stationnarité des patterns.

In [ ]:
df_sorted = df_ml.sort_values('date_creation').reset_index(drop=True)
X_all     = df_sorted[FEATURES].fillna(df_sorted[FEATURES].median())
y_all     = df_sorted[CIBLE]

# TODO : créer TimeSeriesSplit avec 5 folds
tscv       = TimeSeriesSplit(# TODO)
recalls_cv = []
aucs_cv    = []

for fold, (tr_idx, te_idx) in enumerate(tscv.split(X_all)):
    # TODO : entraîner RandomForest léger (n_estimators=100)
    rf_cv = RandomForestClassifier(# TODO)
    rf_cv.fit(X_all.iloc[tr_idx], y_all.iloc[tr_idx])
    # TODO : prédire avec seuil SEUIL_OPTIMAL, calculer recall et AUC
    y_proba_cv = rf_cv.predict_proba(X_all.iloc[te_idx])[:, 1]
    y_pred_cv  = (y_proba_cv >= SEUIL_OPTIMAL).astype(int)
    r  = # TODO
    au = # TODO
    recalls_cv.append(r)
    aucs_cv.append(au)
    print(f'  Fold {fold+1} | Recall: {r:.4f} | AUC: {au:.4f}')

print(f'\nRecall moyen : {np.mean(recalls_cv):.4f}')
print(f'Écart-type   : {np.std(recalls_cv):.4f}')

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
## Étape 8 — Génération du fichier d'alertes

### MÉTHODE
On applique le modèle sur **toutes les livraisons En transit** — ce sont les seules pour lesquelles une intervention est encore possible. Les livraisons déjà livrées ou perdues ne peuvent pas être sauvées.

Le fichier `logitrack_risque_scores.csv` est stratifié en **3 niveaux d'intervention** selon le score de risque — pour permettre aux dispatchers de prioriser leurs actions sans être submergés.

In [ ]:
df_transit = df[df['statut'] == 'En transit'].copy()
if len(df_transit) == 0:
    df_transit = df.tail(500).copy()
    print('⚠️  Mode démo — 500 dernières livraisons')

X_score = df_transit[FEATURES].fillna(df_transit[FEATURES].median())

# TODO : générer les scores avec rf.predict_proba (colonne [:, 1])
scores = # TODO

df_transit = df_transit.copy()
df_transit['score_risque'] = scores
# TODO : créer alerte_retard = 1 si score >= SEUIL_OPTIMAL
df_transit['alerte_retard'] = # TODO

# TODO : créer niveau_urgence
# Critique si > 0.75, Élevé si > 0.55, Modéré sinon (parmi les alertes)
def niveau_urgence(score):
    # TODO
    pass

df_transit['niveau_urgence'] = df_transit['score_risque'].apply(niveau_urgence)

cols_export = ['livraison_id','client_id','transporteur_id',
               'pays_origine','pays_destination','date_creation',
               'date_livraison_prevue','priorite','type_colis',
               'sla_jours_contractuel','risque_douanier',
               'taux_breach_transporteur','taux_breach_corridor',
               'score_risque','alerte_retard','niveau_urgence']
cols_export = [c for c in cols_export if c in df_transit.columns]

df_alertes = (
    df_transit[cols_export]
    .sort_values('score_risque', ascending=False)
    .reset_index(drop=True)
)
df_alertes.to_csv(f'{SAVE_PATH}logitrack_risque_scores.csv', index=False)
print(f'✅ Alertes : {df_alertes["alerte_retard"].sum():,} déclenchées')
print(df_alertes[['livraison_id','pays_origine','pays_destination',
                   'score_risque','niveau_urgence']].head(5).to_string())

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
## Bilan du Notebook 5

| Metric | Logistic Regression | Random Forest | Gradient Boosting |
|---|---|---|---|
| AUC-ROC | ~0.72 | **~0.86** | ~0.85 |
| Recall (seuil déf.) | ~0.82 | ~0.52 | ~0.54 |

| Élément | Valeur |
|---|---|
| Modèle choisi | **Random Forest** (meilleur AUC) |
| Seuil optimal | Variable selon le dataset — satisfait Recall ≥ 0.75 |
| Feature #1 | `taux_breach_transporteur` (~22%) |
| Feature #2 | `taux_breach_corridor` (~18%) |
| Feature #3 | `risque_douanier_num` (~14%) |
| Validation CV | `TimeSeriesSplit` 5 folds — Recall stable ✅ |

**Fichiers exportés dans `SAVE_PATH` :**
- `logitrack_risque_scores.csv` — alertes stratifiées pour Power BI
- `logitrack_precision_recall_curve.png`
- `logitrack_feature_importance.png`

**Recommandation finale :** En production, ré-entraîner le modèle chaque trimestre et surveiller le Recall mensuel. Si Recall < 0.70 deux mois consécutifs → investigation des changements opérationnels récents (nouveau transporteur, nouvelle route, modification SLA).

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.